<a href="https://colab.research.google.com/github/mic006016/ssd-cctv-detection/blob/main/Pytorch_SSD_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 구글 드라이브 연결
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# 구글 드라이브 디렉토리 확인 및 이동
!ls /content/gdrive

MyDrive


In [ ]:
ls

gdrive/  sample_data/


In [ ]:
cd /content/gdrive/MyDrive/ssd/

/content/gdrive/MyDrive/ssd


In [ ]:
import torch
print(torch.__version__)

2.11.0+cu128


In [ ]:
# GPU 사용 체크
is_cuda = False
if torch.cuda.is_available():
  is_cuda = True

print("CUDA 사용 가능 여부:", torch.cuda.is_available())

CUDA 사용 가능 여부: True


In [ ]:
# 파이토치 SSD 실습 소스 파일들 다운로드
!git clone https://github.com/dusty-nv/pytorch-ssd

fatal: destination path 'pytorch-ssd' already exists and is not an empty directory.


In [ ]:
cd pytorch-ssd

/content/gdrive/MyDrive/ssd/pytorch-ssd


In [ ]:
ls

data/                      models/                    __pycache__/
eval_ssd.py                my_ssd_opencv_cctv.py      README.md
inference_ssd_nano_log.py  my_ssd_opencv.py           requirements.txt
inference_ssd_nano.py      onnx_export.py             run_ssd_example.py
inference_ssd_windows.py   open_images_classes.txt    train_ssd.py
LICENSE                    open_images_downloader.py  vision/


In [ ]:
# 파이토치 SSD MobileNetV1 미리 학습된 모델 파일 다운로드
!wget https://nvidia.box.com/shared/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth -O models/mobilenet-v1-ssd-mp-0_675.pth


--2026-06-10 11:14:38--  https://nvidia.box.com/shared/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth
Resolving nvidia.box.com (nvidia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to nvidia.box.com (nvidia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth [following]
--2026-06-10 11:14:39--  https://nvidia.box.com/public/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth
Reusing existing connection to nvidia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://nvidia.app.box.com/public/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth [following]
--2026-06-10 11:14:39--  https://nvidia.app.box.com/public/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth
Resolving nvidia.app.box.com (nvidia.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to nvidia.app.box.com (nvidia.app.box.com)|74.112.186.157|:443... connected.
HTTP request

In [ ]:
# 필수 패키지 설치
!pip3 install -v -r requirements.txt

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Obtaining dependency information for boto3 from https://files.pythonhosted.org/packages/c9/39/06d683934d94972c070179952746a57adff5c1851e2396d505a9b8cd9754/boto3-1.43.26-py3-none-any.whl.metadata
  Obtaining dependency information for botocore<1.44.0,>=1.43.26 from https://files.pythonhosted.org/packages/be/e6/5a5ec1033613e7812e5b19ec8c2a1889834fde336d8812d53019eac6e04a/botocore-1.43.26-py3-none-any.whl.metadata
  Obtaining dependency information for jmespath<2.0.0,>=0.7.1 from https://files.pythonhosted.org/packages/14/2f/967ba146e6d58cf6a652da73885f52fc68001525b4197effc174321d70b4/jmespath-1.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for s3transfer<0.19.0,>=0.18.0 from https://files.pythonhosted.org/packages/2b/58/a58fc997655386daa2e25784e30c288aa3e3819e401f77029ee4899fb55a/s3transfer-0.18.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 12.5 MB

In [ ]:
# CCTV에서 감시하고자 하는 대상인 사람, 차, 버스 데이터를 다운로드
!python3 open_images_downloader.py --class-names "Person, Car, Bus" --data=data/cctv --max-images=4500 --num-workers=2

2026-06-10 11:14:57 - Requested 3 classes, found 3 classes
2026-06-10 11:14:57 - Read annotation file data/cctv/train-annotations-bbox.csv
2026-06-10 11:15:19 - Available train images:  302983
2026-06-10 11:15:19 - Available train boxes:   1174637

2026-06-10 11:15:19 - Read annotation file data/cctv/validation-annotations-bbox.csv
2026-06-10 11:15:20 - Available validation images:  10333
2026-06-10 11:15:20 - Available validation boxes:   21748

2026-06-10 11:15:20 - Read annotation file data/cctv/test-annotations-bbox.csv
2026-06-10 11:15:27 - Available test images:  31117
2026-06-10 11:15:27 - Available test boxes:   66815

2026-06-10 11:15:27 - Total available images: 344433
2026-06-10 11:15:27 - Total available boxes:  1263200

2026-06-10 11:15:27 - Limiting train dataset to:  3958 images (15974 boxes)
2026-06-10 11:15:27 - Limiting validation dataset to:  135 images (287 boxes)
2026-06-10 11:15:27 - Limiting test dataset to:  406 images (925 boxes)

------------------------------

In [ ]:
# 파이토치 SSD MobileNetV1 학습
!python3 train_ssd.py --data=data/cctv --model-dir=models/cctv --batch-size=16 --epochs=45

2026-06-10 11:38:37.637626: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 11:38:37.711424: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-10 11:39:16 - Using CUDA...
2026-06-10 11:39:16 - Namespace(dataset_type='open_images', datasets=['data/cctv'], balance_data=False, net='mb1-ssd', resolution=300, freeze_base_net=False, freeze_net=False, mb2_width_mult=1.0, base_net=None, pretrained_ssd='models/mobilenet-v1-ssd-mp-0_675.pth', resume=None, lr=0.01, momentum=0.9, weight_decay=0.0005, gamma=0.1, base